In [ ]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder, StandardScaler


In [ ]:
csv_files = ["/train_ads.csv", "/train_gmap.csv", "/train_hotel.csv", "/train_rant.csv"]


In [ ]:
df_list=[]
for file in csv_files:
    temp = pd.read_csv(file)
    temp = temp[["review", "review_length", "sentiment", "relevancy_score", "label"]]
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)
df = df.dropna()
df = df[df["review"].str.strip().str.len() > 0]

In [ ]:
allowed_classes = ["VALID", "IRRELEVANT", "AD", "NOT VISITED"]

df = df[df['label'].isin(allowed_classes)].reset_index(drop=True)

print(df['label'].value_counts())

label
VALID          3623
AD             2000
IRRELEVANT     1714
NOT VISITED    1600
Name: count, dtype: int64


In [ ]:
balanced_df_list = []

for label in df['label'].unique():
    class_subset = df[df['label'] == label].sample(n=1600, random_state=42)
    balanced_df_list.append(class_subset)

b_df = pd.concat(balanced_df_list, ignore_index=True)

b_df = b_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(b_df['label'].value_counts())

label
VALID          1600
IRRELEVANT     1600
NOT VISITED    1600
AD             1600
Name: count, dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
train_df, temp_df = train_test_split(
    b_df,
    test_size=0.2,
    stratify=b_df['label'],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=42
)
print("Train label counts:\n", train_df['label'].value_counts())
print("Validation label counts:\n", val_df['label'].value_counts())
print("Test label counts:\n", test_df['label'].value_counts())

Train label counts:
 label
VALID          1280
AD             1280
NOT VISITED    1280
IRRELEVANT     1280
Name: count, dtype: int64
Validation label counts:
 label
IRRELEVANT     160
VALID          160
NOT VISITED    160
AD             160
Name: count, dtype: int64
Test label counts:
 label
IRRELEVANT     160
AD             160
NOT VISITED    160
VALID          160
Name: count, dtype: int64


In [ ]:
le = LabelEncoder()
train_df['label_encoded'] = le.fit_transform(train_df['label'])
val_df['label_encoded'] = le.transform(val_df['label'])
test_df['label_encoded'] = le.transform(test_df['label'])

In [ ]:
scaler = StandardScaler()
num_cols = ["review_length", "sentiment", "relevancy_score"] #numeric columns
train_df[num_cols] = scaler.fit_transform(train_df[num_cols])
val_df[num_cols] = scaler.transform(val_df[num_cols])
test_df[num_cols] = scaler.transform(test_df[num_cols])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(df):
    return tokenizer(df['review'].tolist(), truncation=True, padding=True, return_tensors="pt")

train_enc = tokenize(train_df)
val_enc = tokenize(val_df)
test_enc = tokenize(test_df)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



Using device: cuda


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, encodings, features, labels):
        self.encodings = encodings
        self.features = features
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['features'] = torch.tensor(self.features[idx], dtype=torch.float)
        item['labels'] = torch.tensor(self.labels[idx])
        return item
train_dataset = ReviewDataset(train_enc, train_df[num_cols].values, train_df['label_encoded'].values)
val_dataset = ReviewDataset(val_enc, val_df[num_cols].values, val_df['label_encoded'].values)
test_dataset = ReviewDataset(test_enc, test_df[num_cols].values, test_df['label_encoded'].values)

In [ ]:
class BertWithFeatures(torch.nn.Module):
    def __init__(self, bert_model_name, num_features, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(bert_model_name)
        hidden_size = self.bert.config.hidden_size
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_size + num_features, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, num_labels)
        )
    def forward(self, input_ids, attention_mask, features, labels=None):
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        features = features.to(device)
        if labels is not None:
            labels = labels.to(device)

        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        combined = torch.cat([pooled_output, features], dim=1)
        logits = self.classifier(combined)
        loss = None
        if labels is not None:
            loss_fn = torch.nn.CrossEntropyLoss()
            loss = loss_fn(logits, labels)
        return {"loss": loss, "logits": logits}

model = BertWithFeatures("bert-base-uncased", num_features=len(num_cols), num_labels=4)
model.to(device)

BertWithFeatures(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementw

In [ ]:
def collate_fn(batch):
    input_ids = torch.stack([item['input_ids'] for item in batch])
    attention_mask = torch.stack([item['attention_mask'] for item in batch])
    features = torch.stack([item['features'] for item in batch])
    labels = torch.stack([item['labels'] for item in batch])
    return {"input_ids": input_ids, "attention_mask": attention_mask, "features": features, "labels": labels}




In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro"
    )
    acc = accuracy_score(labels, predictions)
    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    overwrite_output_dir=True,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=2,

    logging_steps=20,
    save_strategy="steps",
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,

    load_best_model_at_end=True,
    metric_for_best_model="f1",          # track macro F1
    greater_is_better=True,

    save_total_limit=2,
    seed=42
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-4185077078.py:28: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()
import wandb

# Save locally first
trainer.save_model("my_model")

# Initialize wandb run (if not already active)
wandb.init(project="tiktok-on-the-clock")

# Log the model folder as a W&B artifact
artifact = wandb.Artifact("my_model", type="model")
artifact.add_dir("my_model")
wandb.log_artifact(artifact)

Step,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
50,0.776900,0.489861,0.989062,0.989072,0.989062,0.989062
100,0.225500,0.185096,0.990625,0.990775,0.990625,0.990624
150,0.126300,0.081646,0.993750,0.993750,0.993750,0.993750
200,0.055400,0.060493,0.993750,0.993788,0.993750,0.993750
250,0.059800,0.047281,0.993750,0.993750,0.993750,0.993750
300,0.081800,0.041414,0.993750,0.993750,0.993750,0.993750
350,0.051200,0.039858,0.993750,0.993750,0.993750,0.993750
400,0.031900,0.046353,0.992188,0.992273,0.992188,0.992187
450,0.031000,0.040137,0.993750,0.993750,0.993750,0.993750
500,0.072900,0.037611,0.993750,0.993788,0.993750,0.993750


eval/accuracy,▁▃█████▆████
eval/f1,▁▃█████▆████
eval/loss,█▃▂▁▁▁▁▁▁▁▁▁
eval/precision,▁▄█████▆████
eval/recall,▁▃█████▆████
eval/runtime,▃▁▅▃▅▃█▅▇▃▃▃
eval/samples_per_second,▆█▄▅▄▆▁▄▂▆▆▆
eval/steps_per_second,▆█▄▅▄▆▁▄▂▆▆▆
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇████
train/grad_norm,▂▂▂▂▁▁▂▁▁▁▁▁▅▁▂▁▁▁▁▁▄▁▁▁█▁▁▁▁▁▁▁


wandb: Adding directory to artifact (./my_model)... Done. 4.2s


<Artifact my_model>

In [ ]:
import torch
import wandb

wandb.init()
torch.save(model.state_dict(), "bert_with_features.pt")
wandb.save("bert_with_features.pt")

['/content/wandb/run-20250830_225926-f3daq1tu/files/bert_with_features.pt']

In [ ]:
import torch
from transformers import AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Load your trained model
model = BertWithFeatures("bert-base-uncased", num_features=3, num_labels=4)  # change 5 to your num_features
model.load_state_dict(torch.load("bert_with_features.pt", map_location=device))
model.to(device)
model.eval()

def predict_review(text, feature_list):
    """
    text: string, the review text
    feature_list: list or 1D array of extra features
    """
    # Tokenize
    encoding = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    )

    input_ids = encoding["input_ids"].to(device)
    attention_mask = encoding["attention_mask"].to(device)

    # Features tensor
    features = torch.tensor([feature_list], dtype=torch.float).to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask, features)
        logits = outputs["logits"]
        print("Logits:", logits)
        predicted_class = torch.argmax(logits, dim=1).item()

    return predicted_class

In [ ]:
print(dict(zip(le.classes_, le.transform(le.classes_))))

{'AD': np.int64(0), 'IRRELEVANT': np.int64(1), 'NOT VISITED': np.int64(2), 'VALID': np.int64(3)}


In [ ]:
text = " click on this link for good food"

features = [21,-0.3125,0]  # your extra features

predicted_class = predict_review(text, features)
print("Predicted class:", predicted_class)

Logits: tensor([[-0.9495, -1.6224, -0.7129,  2.1634]], device='cuda:0')
Predicted class: 3


In [ ]:
# ================= SAVE MODEL SAU KHI TRAIN =================
import joblib

save_dir = "./saved_model"
os.makedirs(save_dir, exist_ok=True)

# Save tokenizer + BERT backbone
model.bert.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

# Save toàn bộ model (bao gồm classifier)
torch.save(model.state_dict(), os.path.join(save_dir, "pytorch_model.bin"))

# Save thêm label encoder và scaler để preprocessing
joblib.dump(le, os.path.join(save_dir, "label_encoder.pkl"))
joblib.dump(scaler, os.path.join(save_dir, "scaler.pkl"))

print(f"✅ Model, tokenizer, label encoder, scaler đã được lưu tại {save_dir}")

from transformers import AutoTokenizer, AutoModel
import torch
import joblib

# Load lại tokenizer + BERT backbone
tokenizer = AutoTokenizer.from_pretrained("./saved_model")
bert = AutoModel.from_pretrained("./saved_model")

# Load classifier
model = BertWithFeatures("bert-base-uncased", num_features=3, num_labels=4)
model.load_state_dict(torch.load("./saved_model/pytorch_model.bin", map_location="cpu"))
model.eval()

# Load label encoder và scaler
le = joblib.load("./saved_model/label_encoder.pkl")
scaler = joblib.load("./saved_model/scaler.pkl")

✅ Model, tokenizer, label encoder, scaler đã được lưu tại ./saved_model


In [ ]:
import pandas as pd
# Access data from the test_dataset
# Example: numeric features for each review
numeric_features = ['review_length', 'sentiment', 'relevancy_score']  # must match model's num_features

# Extract the numeric features and labels from the test_dataset
# Convert the dataset to a list of dictionaries to easily access the data
test_data_list = [test_dataset[i] for i in range(len(test_dataset))]

# Extract the numeric features and true labels
X_numeric = torch.stack([item['features'] for item in test_data_list]).numpy()
y_true = torch.stack([item['labels'] for item in test_data_list]).numpy()

/tmp/ipython-input-993723768.py:9: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  item = {key: val[idx] for key, val in self.encodings.items()}


IndexError: too many indices for tensor of dimension 2